In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json, explode_outer

is_schema = ArrayType(StructType([
    StructField("date", StringType()),
    StructField("symbol", StringType()),
    StructField("reportedCurrency", StringType()),
    StructField("cik", StringType()),
    StructField("filingDate", StringType()),
    StructField("acceptedDate", StringType()),
    StructField("fiscalYear", StringType()),
    StructField("period", StringType()),
    StructField("revenue", LongType()),
    StructField("costOfRevenue", LongType()),
    StructField("grossProfit", LongType()),
    StructField("researchAndDevelopmentExpenses", LongType()),
    StructField("generalAndAdministrativeExpenses", LongType()),
    StructField("sellingAndMarketingExpenses", LongType()),
    StructField("sellingGeneralAndAdministrativeExpenses", LongType()),
    StructField("otherExpenses", LongType()),
    StructField("operatingExpenses", LongType()),
    StructField("costAndExpenses", LongType()),
    StructField("netInterestIncome", LongType()),
    StructField("interestIncome", LongType()),
    StructField("interestExpense", LongType()),
    StructField("depreciationAndAmortization", LongType()),
    StructField("ebitda", LongType()),
    StructField("ebit", LongType()),
    StructField("nonOperatingIncomeExcludingInterest", LongType()),
    StructField("operatingIncome", LongType()),
    StructField("totalOtherIncomeExpensesNet", LongType()),
    StructField("incomeBeforeTax", LongType()),
    StructField("incomeTaxExpense", LongType()),
    StructField("netIncomeFromContinuingOperations", LongType()),
    StructField("netIncomeFromDiscontinuedOperations", LongType()),
    StructField("otherAdjustmentsToNetIncome", LongType()),
    StructField("netIncome", LongType()),
    StructField("netIncomeDeductions", LongType()),
    StructField("bottomLineNetIncome", LongType()),
    StructField("eps", DoubleType()),
    StructField("epsDiluted", DoubleType()),
    StructField("weightedAverageShsOut", LongType()),
    StructField("weightedAverageShsOutDil", LongType())
]))

bs_schema = ArrayType(StructType([
    StructField("date", StringType()),
    StructField("symbol", StringType()),
    StructField("reportedCurrency", StringType()),
    StructField("cik", StringType()),
    StructField("filingDate", StringType()),
    StructField("acceptedDate", StringType()),
    StructField("fiscalYear", StringType()),
    StructField("period", StringType()),
    StructField("cashAndCashEquivalents", LongType()),
    StructField("shortTermInvestments", LongType()),
    StructField("cashAndShortTermInvestments", LongType()),
    StructField("netReceivables", LongType()),
    StructField("accountsReceivables", LongType()),
    StructField("otherReceivables", LongType()),
    StructField("inventory", LongType()),
    StructField("prepaids", LongType()),
    StructField("otherCurrentAssets", LongType()),
    StructField("totalCurrentAssets", LongType()),
    StructField("propertyPlantEquipmentNet", LongType()),
    StructField("goodwill", LongType()),
    StructField("intangibleAssets", LongType()),
    StructField("goodwillAndIntangibleAssets", LongType()),
    StructField("longTermInvestments", LongType()),
    StructField("taxAssets", LongType()),
    StructField("otherNonCurrentAssets", LongType()),
    StructField("totalNonCurrentAssets", LongType()),
    StructField("otherAssets", LongType()),
    StructField("totalAssets", LongType()),
    StructField("totalPayables", LongType()),
    StructField("accountPayables", LongType()),
    StructField("otherPayables", LongType()),
    StructField("accruedExpenses", LongType()),
    StructField("shortTermDebt", LongType()),
    StructField("capitalLeaseObligationsCurrent", LongType()),
    StructField("taxPayables", LongType()),
    StructField("deferredRevenue", LongType()),
    StructField("otherCurrentLiabilities", LongType()),
    StructField("totalCurrentLiabilities", LongType()),
    StructField("longTermDebt", LongType()),
    StructField("capitalLeaseObligationsNonCurrent", LongType()),
    StructField("deferredRevenueNonCurrent", LongType()),
    StructField("deferredTaxLiabilitiesNonCurrent", LongType()),
    StructField("otherNonCurrentLiabilities", LongType()),
    StructField("totalNonCurrentLiabilities", LongType()),
    StructField("otherLiabilities", LongType()),
    StructField("capitalLeaseObligations", LongType()),
    StructField("totalLiabilities", LongType()),
    StructField("treasuryStock", LongType()),
    StructField("preferredStock", LongType()),
    StructField("commonStock", LongType()),
    StructField("retainedEarnings", LongType()),
    StructField("additionalPaidInCapital", LongType()),
    StructField("accumulatedOtherComprehensiveIncomeLoss", LongType()),
    StructField("otherTotalStockholdersEquity", LongType()),
    StructField("totalStockholdersEquity", LongType()),
    StructField("totalEquity", LongType()),
    StructField("minorityInterest", LongType()),
    StructField("totalLiabilitiesAndTotalEquity", LongType()),
    StructField("totalInvestments", LongType()),
    StructField("totalDebt", LongType()),
    StructField("netDebt", LongType())
]))

# Income Statements

is_df = spark.read.table("workspace.stock_data.income_statements_bronze")

is_parsed_df = is_df.withColumn(
    "parsed",
    from_json(col("raw_json"), is_schema)
)

is_exploded_df = (
    is_parsed_df
        .withColumn("profile", explode_outer("parsed"))
)

is_silver_df = is_exploded_df.select(
    col("profile.date"),
    col("profile.symbol"),
    col("profile.reportedCurrency").alias("reported_currency"),
    col("profile.fiscalYear").alias("fiscal_year"),
    col("profile.period"),
    col("profile.revenue"),
    col("profile.grossProfit").alias("gross_profit"),
    col("profile.interestExpense").alias("interest_expense"),
    col("profile.ebitda"),
    col("profile.ebit"),
    col("profile.netIncome").alias("net_income"),
    col("profile.eps")
)

cleaned_is_silver_df = (
    is_silver_df
        .dropDuplicates(["symbol", "fiscal_year", "period"])
        .filter((col('date') >= '2015-12-31') & (col('revenue') > 0))
)

cleaned_is_silver_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.income_statements_silver')

# Balance Sheets

bs_df = spark.read.table("workspace.stock_data.balance_sheets_bronze")

bs_parsed_df = bs_df.withColumn(
    "parsed",
    from_json(col("raw_json"), bs_schema))

bs_exploded_df = (
    bs_parsed_df
        .withColumn("profile", explode_outer("parsed")))

bs_silver_df = bs_exploded_df.select(
    col("profile.date"),
    col("profile.symbol"),
    col("profile.reportedCurrency").alias("reported_currency"),
    col("profile.fiscalYear").alias("fiscal_year"),
    col("profile.period"),
    col("profile.totalAssets").alias("total_assets"),
    col("profile.totalLiabilities").alias("total_liabilities"),
    col("profile.totalEquity").alias("total_equity"),
    col("profile.totalDebt").alias("total_debt"),
    col("profile.totalCurrentAssets").alias("total_current_assets"),
    col("profile.totalCurrentLiabilities").alias("total_current_liabilities")
)

cleaned_bs_silver_df = (
    bs_silver_df
        .dropDuplicates(["symbol", "fiscal_year", "period"])
        .filter(((col('date') >= '2015-12-31') & (col('total_assets') == col('total_liabilities') + col('total_equity'))))
)

cleaned_bs_silver_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.balance_sheets_silver')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.income_statements_silver

In [0]:
%sql
SELECT *
FROM workspace.stock_data.balance_sheets_silver